# TilingDataModuleWrapper Tutorial

This notebook shows how the wrapper works end-to-end: wrapping a base datamodule, producing tiled batches, and stitching tile predictions back to full-image outputs.

## What You Will Learn
1. How to wrap a base datamodule
2. How tiling is controlled per split
3. How to inspect tiled train/predict batches
4. How to stitch tile predictions

## How The Wrapper Works

`TilingDataModuleWrapper` wraps any existing Lightning datamodule and intercepts dataloaders.

- Delegates `prepare_data()` and `setup()` to the base datamodule
- Wraps selected split datasets (`train`, `val`, `test`, `predict`) with `TiledDataset`
- Uses `tile_size`, `overlap`, and caching settings from wrapper configuration
- Exposes tile metadata required for reconstruction and stitching

In [1]:
import tempfile
import torch
from torch.utils.data import Dataset, DataLoader
from lightning import LightningDataModule

from terratorch.datamodules import TilingDataModuleWrapper

print("PyTorch version:", torch.__version__)

/Users/ahmedemam/miniconda3/envs/ttorch-env-311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0405 10:29:09.688000 64497 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/ahmedemam/miniconda3/envs/ttorch-env-311/lib/python3.11/site-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
/Users/ahmedemam/miniconda3/envs/ttorch-env-311/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


PyTorch version: 2.9.1


## Create A Minimal Base DataModule

The wrapper is generic: any Lightning datamodule can be wrapped as long as it returns dataloaders with dataset samples the wrapper can tile.

In [2]:
class SimpleDataset(Dataset):
    def __init__(self, num_samples=3, image_size=(640, 640), num_classes=4):
        self.num_samples = num_samples
        self.image_size = image_size
        self.num_classes = num_classes

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        h, w = self.image_size
        image = torch.rand(3, h, w)
        mask = torch.randint(0, self.num_classes, (h, w))
        return {
            "image": image,
            "mask": mask,
            "filename": f"sample_{idx}.tif"
        }


class SimpleDataModule(LightningDataModule):
    def __init__(self, batch_size=2, num_workers=0):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_dataset = SimpleDataset(num_samples=3, image_size=(640, 640))
        self.predict_dataset = SimpleDataset(num_samples=2, image_size=(768, 768))

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, num_workers=self.num_workers, shuffle=False)

    def predict_dataloader(self):
        return DataLoader(self.predict_dataset, batch_size=self.batch_size, num_workers=self.num_workers, shuffle=False)


base_dm = SimpleDataModule(batch_size=2, num_workers=0)
base_dm.setup("fit")
print("Base datamodule ready.")

Base datamodule ready.


## Wrap The DataModule And Inspect Tiled Batches

Apply the wrapper to `train` and `predict` splits and inspect the resulting batch structure.

In [3]:
with tempfile.TemporaryDirectory() as cache_dir:
    wrapped_dm = TilingDataModuleWrapper(
        base_datamodule=base_dm,
        tile_size=(256, 256),
        overlap=64,
        cache_dir=cache_dir,
        apply_to_splits=["train", "predict"],
        keep_incomplete_tiles=True
    )

    wrapped_dm.setup("fit")

    train_loader = wrapped_dm.train_dataloader()
    train_batch = next(iter(train_loader))
    print("Train batch keys:", list(train_batch.keys()))
    print("Train image batch shape:", train_batch["image"].shape)

    pred_loader = wrapped_dm.predict_dataloader()
    pred_batch = next(iter(pred_loader))
    print("Predict batch keys:", list(pred_batch.keys()))
    print("Predict image batch shape:", pred_batch["image"].shape)

    if "tile_coords" in pred_batch:
        print("Example tile coordinates:", pred_batch["tile_coords"][0])

2026-04-05 10:29:13,271 - INFO - [TilingDataModuleWrapper] Created with tile_size=(256, 256), overlap=64, cache_dir=/var/folders/bd/fm8q6wy5563f7z_lpc5mm4vw0000gn/T/tmpynknhkxm, splits=['train', 'predict']
2026-04-05 10:29:13,272 - INFO - [TilingDataModuleWrapper] Wrapping 'train' dataset with tiling
2026-04-05 10:29:13,272 - INFO - [TiledDataset] Building tile index...
2026-04-05 10:29:13,283 - INFO - [TiledDataset] Built 48 tiles, saved index to /var/folders/bd/fm8q6wy5563f7z_lpc5mm4vw0000gn/T/tmpynknhkxm/train/tile_index_0d97b67f44cc3934a9b951c6a8eeee72.json
2026-04-05 10:29:13,298 - INFO - [TilingDataModuleWrapper] Wrapping 'predict' dataset with tiling
2026-04-05 10:29:13,299 - INFO - [TiledDataset] Building tile index...
2026-04-05 10:29:13,309 - INFO - [TiledDataset] Built 32 tiles, saved index to /var/folders/bd/fm8q6wy5563f7z_lpc5mm4vw0000gn/T/tmpynknhkxm/predict/tile_index_9029dae8c8033c46fdb1aafc3bcd1801.json


Train batch keys: ['image', 'mask', 'filename', 'tile_coords', 'base_idx']
Train image batch shape: torch.Size([2, 3, 256, 256])
Predict batch keys: ['image', 'mask', 'filename', 'tile_coords', 'base_idx']
Predict image batch shape: torch.Size([2, 3, 256, 256])
Example tile coordinates: (0, 0, 256, 256)


## Stitch Predictions Back To Full Image

When using tiled inference, merge tile-level predictions with `stitch_predictions`.

In [4]:
tile_predictions = torch.rand(4, 1, 256, 256)
tile_coords = [
    (0, 0, 256, 256),
    (0, 192, 256, 448),
    (192, 0, 448, 256),
    (192, 192, 448, 448),
]

stitched = TilingDataModuleWrapper.stitch_predictions(
    tile_predictions=tile_predictions,
    tile_coords=tile_coords,
    original_size=(448, 448),
    overlap=64,
    use_blending=True,
)

print("Stitched output shape:", stitched.shape)

Stitched output shape: torch.Size([1, 448, 448])


## Summary

- `TilingDataModuleWrapper` wraps an existing datamodule without changing task code.
- Tiling behavior is configured with `tile_size`, `overlap`, `apply_to_splits`, and caching options.
- Wrapped dataloaders provide tiled batches and metadata for reconstruction workflows.
- `stitch_predictions` merges tile outputs into full-image predictions.

## Additional Resources

- Wrapper implementation: `terratorch/datamodules/tiled_datamodule_wrapper.py`
- Underlying tiled dataset: `terratorch/datasets/tiled_dataset_wrapper.py`